# Data loading and Cleaning

In [99]:
import numpy as np
import pandas as pd
import seaborn as sns
import glob
import os
import joblib
from sklearn.decomposition import PCA
from sklearn.ensemble import IsolationForest
from sklearn.metrics import (
    auc,
    precision_recall_curve,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
)
from sklearn.preprocessing import QuantileTransformer, RobustScaler
from sklearn.model_selection import ParameterGrid

In [2]:
# loading
assets = joblib.load("nids_iso_forest_pipeline.pkl")
iso_model = assets["iso_model"]
selected_thresh = assets["selected_thresh"]
selected_cols = assets["selected_cols"]
scaler = assets["scaler"]

data = np.load("scaled_datasets.npz")
X_train_scaled = data["X_train"]
X_val_scaled = data["X_val"]
X_test_scaled = data["X_test"]
y_train, y_val, y_test = data["y_train"], data["y_val"], data["y_test"]

print("laoding done")

laoding done


In [3]:
fast_raw_file = "../data/raw/merged_raw.parquet"

if os.path.exists(fast_raw_file):
    print("Loading fast merged raw data...")
    df = pd.read_parquet(fast_raw_file)

else:
    print("Loading CSV files for the first time...")
    path = "../data/raw/*.csv"
    all_files = glob.glob(path)

    df = pd.concat((pd.read_csv(f) for f in all_files), ignore_index=True)

    print("Saving merged data to Parquet in raw folder...")
    df.to_parquet(fast_raw_file, index=False)

print(f"Data successfully loaded! Shape: {df.shape}")

Loading fast merged raw data...
Data successfully loaded! Shape: (2830743, 79)


In [4]:
df.shape

(2830743, 79)

In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 2830743 entries, 0 to 2830742
Data columns (total 79 columns):
 #   Column                        Dtype  
---  ------                        -----  
 0    Destination Port             int64  
 1    Flow Duration                int64  
 2    Total Fwd Packets            int64  
 3    Total Backward Packets       int64  
 4   Total Length of Fwd Packets   int64  
 5    Total Length of Bwd Packets  int64  
 6    Fwd Packet Length Max        int64  
 7    Fwd Packet Length Min        int64  
 8    Fwd Packet Length Mean       float64
 9    Fwd Packet Length Std        float64
 10  Bwd Packet Length Max         int64  
 11   Bwd Packet Length Min        int64  
 12   Bwd Packet Length Mean       float64
 13   Bwd Packet Length Std        float64
 14  Flow Bytes/s                  float64
 15   Flow Packets/s               float64
 16   Flow IAT Mean                float64
 17   Flow IAT Std                 float64
 18   Flow IAT Max                 int

In [6]:
df.describe()

c:\Users\user\anaconda3\envs\ai_net_env\Lib\site-packages\pandas\core\nanops.py:1028: RuntimeWarning: invalid value encountered in subtract
  sqr = _ensure_numeric((avg - values) ** 2)
c:\Users\user\anaconda3\envs\ai_net_env\Lib\site-packages\pandas\core\nanops.py:1028: RuntimeWarning: invalid value encountered in subtract
  sqr = _ensure_numeric((avg - values) ** 2)


,Destination Port,Flow Duration,Total Fwd Packets,Total Backward Packets,Total Length of Fwd Packets,Total Length of Bwd Packets,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,...,act_data_pkt_fwd,min_seg_size_forward,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min
count,2.830743e+06,2.830743e+06,2.830743e+06,2.830743e+06,2.830743e+06,2.830743e+06,2.830743e+06,2.830743e+06,2.830743e+06,2.830743e+06,...,2.830743e+06,2.830743e+06,2.830743e+06,2.830743e+06,2.830743e+06,2.830743e+06,2.830743e+06,2.830743e+06,2.830743e+06,2.830743e+06
mean,8.071483e+03,1.478566e+07,9.361160e+00,1.039377e+01,5.493024e+02,1.616264e+04,2.075999e+02,1.871366e+01,5.820194e+01,6.891013e+01,...,5.418218e+00,-2.741688e+03,8.155132e+04,4.113412e+04,1.531825e+05,5.829582e+04,8.316037e+06,5.038439e+05,8.695752e+06,7.920031e+06
std,1.828363e+04,3.365374e+07,7.496728e+02,9.973883e+02,9.993589e+03,2.263088e+06,7.171848e+02,6.033935e+01,1.860912e+02,2.811871e+02,...,6.364257e+02,1.084989e+06,6.485999e+05,3.933815e+05,1.025825e+06,5.770923e+05,2.363008e+07,4.602984e+06,2.436689e+07,2.336342e+07
min,0.000000e+00,-1.300000e+01,1.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,...,0.000000e+00,-5.368707e+08,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
25%,5.300000e+01,1.550000e+02,2.000000e+00,1.000000e+00,1.200000e+01,0.000000e+00,6.000000e+00,0.000000e+00,6.000000e+00,0.000000e+00,...,0.000000e+00,2.000000e+01,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
50%,8.000000e+01,3.131600e+04,2.000000e+00,2.000000e+00,6.200000e+01,1.230000e+02,3.700000e+01,2.000000e+00,3.400000e+01,0.000000e+00,...,1.000000e+00,2.400000e+01,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
75%,4.430000e+02,3.204828e+06,5.000000e+00,4.000000e+00,1.870000e+02,4.820000e+02,8.100000e+01,3.600000e+01,5.000000e+01,2.616295e+01,...,2.000000e+00,3.200000e+01,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
max,6.553500e+04,1.200000e+08,2.197590e+05,2.919220e+05,1.290000e+07,6.554530e+08,2.482000e+04,2.325000e+03,5.940857e+03,7.125597e+03,...,2.135570e+05,1.380000e+02,1.100000e+08,7.420000e+07,1.100000e+08,1.100000e+08,1.200000e+08,7.690000e+07,1.200000e+08,1.200000e+08


In [7]:
df.columns.tolist()

[' Destination Port',
 ' Flow Duration',
 ' Total Fwd Packets',
 ' Total Backward Packets',
 'Total Length of Fwd Packets',
 ' Total Length of Bwd Packets',
 ' Fwd Packet Length Max',
 ' Fwd Packet Length Min',
 ' Fwd Packet Length Mean',
 ' Fwd Packet Length Std',
 'Bwd Packet Length Max',
 ' Bwd Packet Length Min',
 ' Bwd Packet Length Mean',
 ' Bwd Packet Length Std',
 'Flow Bytes/s',
 ' Flow Packets/s',
 ' Flow IAT Mean',
 ' Flow IAT Std',
 ' Flow IAT Max',
 ' Flow IAT Min',
 'Fwd IAT Total',
 ' Fwd IAT Mean',
 ' Fwd IAT Std',
 ' Fwd IAT Max',
 ' Fwd IAT Min',
 'Bwd IAT Total',
 ' Bwd IAT Mean',
 ' Bwd IAT Std',
 ' Bwd IAT Max',
 ' Bwd IAT Min',
 'Fwd PSH Flags',
 ' Bwd PSH Flags',
 ' Fwd URG Flags',
 ' Bwd URG Flags',
 ' Fwd Header Length',
 ' Bwd Header Length',
 'Fwd Packets/s',
 ' Bwd Packets/s',
 ' Min Packet Length',
 ' Max Packet Length',
 ' Packet Length Mean',
 ' Packet Length Std',
 ' Packet Length Variance',
 'FIN Flag Count',
 ' SYN Flag Count',
 ' RST Flag Count',
 ' P

In [8]:
# delete any space and turn features to lowercase
df.columns = df.columns.str.strip().str.lower()

In [9]:
df.columns.tolist()

['destination port',
 'flow duration',
 'total fwd packets',
 'total backward packets',
 'total length of fwd packets',
 'total length of bwd packets',
 'fwd packet length max',
 'fwd packet length min',
 'fwd packet length mean',
 'fwd packet length std',
 'bwd packet length max',
 'bwd packet length min',
 'bwd packet length mean',
 'bwd packet length std',
 'flow bytes/s',
 'flow packets/s',
 'flow iat mean',
 'flow iat std',
 'flow iat max',
 'flow iat min',
 'fwd iat total',
 'fwd iat mean',
 'fwd iat std',
 'fwd iat max',
 'fwd iat min',
 'bwd iat total',
 'bwd iat mean',
 'bwd iat std',
 'bwd iat max',
 'bwd iat min',
 'fwd psh flags',
 'bwd psh flags',
 'fwd urg flags',
 'bwd urg flags',
 'fwd header length',
 'bwd header length',
 'fwd packets/s',
 'bwd packets/s',
 'min packet length',
 'max packet length',
 'packet length mean',
 'packet length std',
 'packet length variance',
 'fin flag count',
 'syn flag count',
 'rst flag count',
 'psh flag count',
 'ack flag count',
 'ur

In [10]:
df["label"].value_counts()

label
BENIGN                        2273097
DoS Hulk                       231073
PortScan                       158930
DDoS                           128027
DoS GoldenEye                   10293
FTP-Patator                      7938
SSH-Patator                      5897
DoS slowloris                    5796
DoS Slowhttptest                 5499
Bot                              1966
Web Attack � Brute Force         1507
Web Attack � XSS                  652
Infiltration                       36
Web Attack � Sql Injection         21
Heartbleed                         11
Name: count, dtype: int64

In [11]:
class_percentage = df["label"].value_counts(normalize = True)* 100
print(class_percentage)

label
BENIGN                        80.300366
DoS Hulk                       8.162981
PortScan                       5.614427
DDoS                           4.522735
DoS GoldenEye                  0.363615
FTP-Patator                    0.280421
SSH-Patator                    0.208320
DoS slowloris                  0.204752
DoS Slowhttptest               0.194260
Bot                            0.069452
Web Attack � Brute Force       0.053237
Web Attack � XSS               0.023033
Infiltration                   0.001272
Web Attack � Sql Injection     0.000742
Heartbleed                     0.000389
Name: proportion, dtype: float64


In [12]:
df.duplicated().sum()

np.int64(308381)

In [13]:
duplicate_rows = df[df.duplicated()]

In [14]:
duplicate_rows

,destination port,flow duration,total fwd packets,total backward packets,total length of fwd packets,total length of bwd packets,fwd packet length max,fwd packet length min,fwd packet length mean,fwd packet length std,...,min_seg_size_forward,active mean,active std,active max,active min,idle mean,idle std,idle max,idle min,label
2109,80,77,2,0,12,0,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
2257,443,3,2,0,12,0,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
2749,443,49,2,0,12,0,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
2862,443,4,2,0,12,0,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
2877,443,1,2,0,12,0,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2830701,53,179,2,2,46,46,23,23,23.0,0.0,...,32,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
2830725,53,161,2,2,82,114,41,41,41.0,0.0,...,32,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
2830726,53,212,2,2,84,162,42,42,42.0,0.0,...,32,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
2830731,443,3,2,0,12,0,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN


In [15]:
for label_name in df["label"].unique():
    class_subset = df[df["label"] == label_name]

    dups_count = class_subset.duplicated().sum()
    total_count = len(class_subset)

    print(
        f"Class: [{label_name}] --> Total: {total_count} rows | Duplicates: {dups_count} rows"
    )

Class: [BENIGN] --> Total: 2273097 rows | Duplicates: 176613 rows
Class: [DDoS] --> Total: 128027 rows | Duplicates: 11 rows
Class: [PortScan] --> Total: 158930 rows | Duplicates: 68111 rows
Class: [Bot] --> Total: 1966 rows | Duplicates: 13 rows
Class: [Infiltration] --> Total: 36 rows | Duplicates: 0 rows
Class: [Web Attack � Brute Force] --> Total: 1507 rows | Duplicates: 37 rows
Class: [Web Attack � XSS] --> Total: 652 rows | Duplicates: 0 rows
Class: [Web Attack � Sql Injection] --> Total: 21 rows | Duplicates: 0 rows
Class: [FTP-Patator] --> Total: 7938 rows | Duplicates: 2005 rows
Class: [SSH-Patator] --> Total: 5897 rows | Duplicates: 2678 rows
Class: [DoS slowloris] --> Total: 5796 rows | Duplicates: 411 rows
Class: [DoS Slowhttptest] --> Total: 5499 rows | Duplicates: 271 rows
Class: [DoS Hulk] --> Total: 231073 rows | Duplicates: 58224 rows
Class: [DoS GoldenEye] --> Total: 10293 rows | Duplicates: 7 rows
Class: [Heartbleed] --> Total: 11 rows | Duplicates: 0 rows


In [16]:
df_clean = df.copy()

In [17]:
benign_class_name = "BENIGN"  

df_benign = df_clean[df_clean["label"] == benign_class_name]
df_attacks = df_clean[df_clean["label"] != benign_class_name]

print(f"Benign: {len(df_benign)}")
print(f"Attack: {len(df_attacks)}")

Benign: 2273097
Attack: 557646


In [18]:
df_clean.duplicated().sum()

np.int64(308381)

In [19]:
df_clean = df_clean.drop_duplicates()

In [20]:
df_clean.duplicated().sum()

np.int64(0)

In [21]:
# Function to Inspect Nulls and Inf Values
def inspect_inf_and_nan(df, label_col="label"):
    """Fast vectorized inspection for Inf and NaN values in dataframe features."""
    df_temp = df.copy()
    report = []
    total_rows = len(df_temp)

    # Select feature columns (exclude label column)
    feature_cols = [col for col in df_temp.columns if col != label_col]

    for col in feature_cols:
        # Fast vectorized numeric conversion
        num_series = pd.to_numeric(df_temp[col], errors="coerce")
        n_inf = np.isinf(num_series).sum()
        n_nan = num_series.isna().sum()

        if n_inf > 0 or n_nan > 0:
            report.append(
                {
                    "column": col,
                    "inf_count": n_inf,
                    "inf_%": round((n_inf / total_rows) * 100, 4),
                    "nan_count": n_nan,
                    "nan_%": round((n_nan / total_rows) * 100, 4),
                    "total_corrupted": n_inf + n_nan,
                }
            )

    report_df = pd.DataFrame(report)
    print("=" * 65)
    print("Report For Nulls And Infinity Values")
    print("=" * 65)

    if not report_df.empty:
        report_df = report_df.sort_values(by="total_corrupted", ascending=False)
        print(report_df.to_string(index=False))
    else:
        print("There are no NaN or Inf values found.")

    return report_df

In [22]:
report_df = inspect_inf_and_nan(df_clean, label_col="label")

Report For Nulls And Infinity Values
        column  inf_count  inf_%  nan_count  nan_%  total_corrupted
  flow bytes/s       1211  0.048        353  0.014             1564
flow packets/s       1564  0.062          0  0.000             1564


In [23]:
# ب. معالجة وتوحيد تحويل الأعمدة العدديدية وتحديد الـ Infs كـ NaNs شمولية
numeric_cols = [col for col in df_clean.columns if col != "label"]
for col in numeric_cols:
    df_clean[col] = pd.to_numeric(df_clean[col], errors="coerce")

# ج. تحديد قناع شامل لكل الصفوف التالفة (NaN أو Inf) عبر جميع الميزات
corrupted_mask = df_clean[numeric_cols].isna().any(axis=1) | np.isinf(
    df_clean[numeric_cols]
).any(axis=1)

In [24]:
# numbers of Nulls in each class in columne = label
print(df_clean[corrupted_mask]["label"].value_counts())

label
BENIGN         1427
PortScan        125
Bot               5
DoS Hulk          3
DDoS              2
FTP-Patator       2
Name: count, dtype: int64


In [25]:
df_clean['label'].value_counts()

label
BENIGN                        2096484
DoS Hulk                       172849
DDoS                           128016
PortScan                        90819
DoS GoldenEye                   10286
FTP-Patator                      5933
DoS slowloris                    5385
DoS Slowhttptest                 5228
SSH-Patator                      3219
Bot                              1953
Web Attack � Brute Force         1470
Web Attack � XSS                  652
Infiltration                       36
Web Attack � Sql Injection         21
Heartbleed                         11
Name: count, dtype: int64

In [26]:
print("Percentage Loss Per Class Before Dropping:")

total_per_class = df_clean["label"].value_counts()
corrupted_per_class = df_clean[corrupted_mask]["label"].value_counts()
percentage_loss = (corrupted_per_class / total_per_class) * 100
print(percentage_loss.dropna().round(4).astype(str) + " %")

Percentage Loss Per Class Before Dropping:
label
BENIGN         0.0681 %
Bot             0.256 %
DDoS           0.0016 %
DoS Hulk       0.0017 %
FTP-Patator    0.0337 %
PortScan       0.1376 %
Name: count, dtype: str


In [27]:
# Now we can remove nulls and we are reassured
# need to handle those tow columnes in Online Preprocessing Pipeline

In [28]:
df_clean.replace([np.inf, -np.inf], np.nan, inplace=True)
print(f"number of raws before drop: {len(df_clean):,}")

df_clean.dropna(inplace=True)
df_clean.reset_index(drop=True, inplace=True)

print(f"Number of rows after dropping nulls/infs: {len(df_clean):,}")

number of raws before drop: 2,522,362
Number of rows after dropping nulls/infs: 2,520,798


In [29]:
print(f"number of NaNs is {2522362 - 2520798}")

number of NaNs is 1564


# splitting

In [66]:
from sklearn.model_selection import train_test_split
y = df_clean['label'].apply(lambda x: 0 if str(x).strip().upper() == 'BENIGN' else 1)

In [67]:
if df_clean["fwd header length"].equals(df_clean["fwd header length.1"]): 
    print("YES") 
else: 
    print("NO")

YES


In [68]:
cols_to_drop = ['label', 'destination port', 'fwd header length.1']
X = df_clean.drop(columns=[col for col in cols_to_drop if col in df_clean.columns])

In [69]:
# (Stratified Train/Val/Test Split) 
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y, test_size=0.15, random_state=42, stratify=y
)

In [70]:
# Train (70%) and Validation (15%)
# 15/85 = .17647
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val, test_size=0.17647, random_state=42, stratify=y_train_val
)

In [71]:
print(f" {X_train.shape}")
print(f" {X_val.shape}")
print(f" {X_test.shape}")

 (1764559, 76)
 (378119, 76)
 (378120, 76)


In [82]:
X_train_prep = X_train.copy()
X_val_prep = X_val.copy()
X_test_prep = X_test.copy()

# Feature Selection

In [90]:
from sklearn.feature_selection import mutual_info_classif
from sklearn.ensemble import RandomForestClassifier
from boruta import BorutaPy

# Pearson Correlation
def remove_pearson_redundant_features(X_data, threshold):
    corr_matrix = X_data.corr(method="pearson").abs()
    # take the upper triangle
    upper_tri = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))

    # Identify columns whose correlation exceeds the threshold
    to_drop = [
        column for column in upper_tri.columns 
        if any(upper_tri[column] > threshold)
    ]
    df_reduced = X_data.drop(columns=to_drop)

    print(
        f"were excluded {len(to_drop)} / reamining : {df_reduced.shape[1]}"
    )
    return df_reduced, to_drop

In [91]:
# MI-Boruta Hybrid Selection
def run_mi_boruta_feature_selection(
    X_train, y_train, mi_threshold=0.05, sample_size=150000, max_iter=50
):
    """Executes MI-Boruta Hybrid Feature Selection combining Mutual Information 
    filtering with Boruta shadow feature testing.
    """    
    # Perform fast stratified subsampling if dataset exceeds sample_size
    if len(X_train) > sample_size:
        print(f" Subsampling {sample_size:,} records from training data for fast evaluation...")
        X_sample, _, y_sample, _ = train_test_split(
            X_train, y_train, train_size=sample_size, stratify=y_train, random_state=42
        )
    else:
        X_sample, y_sample = X_train, y_train

    #  Filter features using Mutual Information
    print(f"\n Calculating Mutual Information (filtering scores <= {mi_threshold})...")
    mi_scores = mutual_info_classif(
        X_sample, y_sample, discrete_features="auto", random_state=42
    )

    mi_series = pd.Series(mi_scores, index=X_sample.columns).sort_values(ascending=False)
    mi_passed_cols = mi_series[mi_series > mi_threshold].index.tolist()
    print(f"  Features passing MI pre-filter: {len(mi_passed_cols)} / {X_train.shape[1]}")

    # Fallback safety: If MI filters out too many features, retain top 25 features
    if len(mi_passed_cols) < 5:
        print(" Warning: Too few features passed MI threshold. Falling back to top 25 MI features.")
        mi_passed_cols = mi_series.head(25).index.tolist()

    X_mi_subset = X_sample[mi_passed_cols].copy()

    # Boruta Wrapper selection with Shadow Features
    print("\n Running Boruta Wrapper with Random Forest & Shadow Features...")
    rf = RandomForestClassifier(
        n_estimators=100,
        max_depth=7,
        class_weight="balanced",
        n_jobs=-1,
        random_state=42,
    )

    boruta_selector = BorutaPy(
        estimator=rf,
        n_estimators="auto",
        alpha=0.01, 
        perc=90,
        max_iter=max_iter,
        random_state=42,
        verbose=2,
    )

    boruta_selector.fit(X_mi_subset.values, y_sample.values)

    # Extract confirmed features
    confirmed_mask = boruta_selector.support_
    selected_cols = np.array(mi_passed_cols)[confirmed_mask].tolist()

    # Fallback safety: Include tentative (support_weak_) features if confirmed set is small
    if len(selected_cols) < 5:
        tentative_mask = boruta_selector.support_weak_
        tentative_cols = np.array(mi_passed_cols)[tentative_mask].tolist()
        selected_cols = list(set(selected_cols + tentative_cols))
        print(f" Included tentative Boruta features. Total selected: {len(selected_cols)}")

    # Ultimate fallback safety: If Boruta selects 0 features, use top 15 MI features
    if len(selected_cols) == 0:
        print(" Warning: Boruta confirmed 0 features. Falling back to top 15 MI features.")
        selected_cols = mi_series.head(15).index.tolist()

    print(f"\n MI-Boruta Selection Complete! Selected {len(selected_cols)} features:")
    for i, col in enumerate(selected_cols, 1):
        print(f"  {i}. {col}")

    return selected_cols

In [92]:
# Apply Pearson Redundancy Filter on Training Set Only
X_train_pearson, dropped_pearson = remove_pearson_redundant_features(
    X_train_prep, threshold=0.9
)

# Apply MI-Boruta Hybrid Feature Selection
selected_cols = run_mi_boruta_feature_selection(
    X_train_pearson, y_train, mi_threshold=0.05, sample_size=150000, max_iter=50
)

were excluded 30 / reamining : 46
 Subsampling 150,000 records from training data for fast evaluation...

 Calculating Mutual Information (filtering scores <= 0.05)...
  Features passing MI pre-filter: 27 / 46

 Running Boruta Wrapper with Random Forest & Shadow Features...
Iteration: 	1 / 50
Confirmed: 	0
Tentative: 	27
Rejected: 	0
Iteration: 	2 / 50
Confirmed: 	0
Tentative: 	27
Rejected: 	0
Iteration: 	3 / 50
Confirmed: 	0
Tentative: 	27
Rejected: 	0
Iteration: 	4 / 50
Confirmed: 	0
Tentative: 	27
Rejected: 	0
Iteration: 	5 / 50
Confirmed: 	0
Tentative: 	27
Rejected: 	0
Iteration: 	6 / 50
Confirmed: 	0
Tentative: 	27
Rejected: 	0
Iteration: 	7 / 50
Confirmed: 	0
Tentative: 	27
Rejected: 	0
Iteration: 	8 / 50
Confirmed: 	0
Tentative: 	27
Rejected: 	0
Iteration: 	9 / 50
Confirmed: 	0
Tentative: 	27
Rejected: 	0
Iteration: 	10 / 50
Confirmed: 	27
Tentative: 	0
Rejected: 	0


BorutaPy finished running.

Iteration: 	11 / 50
Confirmed: 	27
Tentative: 	0
Rejected: 	0

 MI-Boruta Selection 

# scaling

In [ ]:
X_train_sub = (
    X_train_prep[selected_cols]
    .copy()
    .fillna(X_train_prep[selected_cols].median(), inplace=True)
)
X_val_sub = (
    X_val_prep[selected_cols]
    .copy()
    .fillna(X_train_prep[selected_cols].median(), inplace=True)
)
X_test_sub = (
    X_test_prep[selected_cols]
    .copy()
    .fillna(X_train_prep[selected_cols].median(), inplace=True)
)

In [114]:
def apply_log_transform(x_tr, x_va, x_te):
    # identify highly skewed features on X_train (Skewness > 2.0)
    x_tr_log = x_tr.copy()
    x_va_log = x_va.copy()
    x_te_log = x_te.copy()
    skewness_scores = x_tr_log.skew().abs()
    high_skew_cols = skewness_scores[skewness_scores > 2].index.tolist()

    all_skewness = x_tr.skew().abs()
    high_skew_cols = all_skewness[all_skewness > 2].index.tolist()

    forced_keywords = [
        "bytes",
        "packets",
        "duration",
        "iat",
        "length",
        "size",
        "active",
        "idle",
    ]
    forced_log_cols = [
        col
        for col in x_tr_log.columns
        if any(k in col.lower() for k in forced_keywords)
    ]

    final_log_cols = list(set(high_skew_cols + forced_log_cols))
    print(
        f" [Log Transform] Applying log1p to {len(final_log_cols)} skewed features."
    )

    final_log_cols = list(set(high_skew_cols + forced_log_cols))
    for col in final_log_cols:
        x_tr_log[col] = np.log1p(x_tr_log[col].clip(lower=0))
        x_va_log[col] = np.log1p(x_va_log[col].clip(lower=0))
        x_te_log[col] = np.log1p(x_te_log[col].clip(lower=0))

    return x_tr_log, x_va_log, x_te_log

In [ ]:
df_clean['fin flag count'].value_counts()

fin flag count
0    2439892
1      80906
Name: count, dtype: int64

In [ ]:
# Robust Pre-Scaling, Dynamic Log1p Transform & Robust Scaling Pipeline
# Subset selected features
# X_train_sub = X_train_prep[selected_cols].copy()
# X_val_sub = X_val_prep[selected_cols].copy()
# X_test_sub = X_test_prep[selected_cols].copy()

In [119]:
#  Coerce data types and replace extreme out-of-bounds float values with NaN
max_float_limit = np.finfo(np.float64).max / 10

for df_sub in [X_train_sub, X_val_sub, X_test_sub]:
    for col in df_sub.columns:
        df_sub[col] = pd.to_numeric(df_sub[col], errors="coerce")
        df_sub[col] = df_sub[col].mask(np.abs(df_sub[col]) >= max_float_limit, np.nan)
print("max num is: ", np.nanmax(X_train_sub.values))

max num is:  2071000000.0


In [ ]:
# Impute missing/corrupted values using X_train medians (Prevents Data Leakage)
# train_medians = X_train_sub.median().fillna(0)

# X_train_sub.fillna(train_medians, inplace=True)
# X_val_sub.fillna(train_medians, inplace=True)
# X_test_sub.fillna(train_medians, inplace=True)

,total length of fwd packets,bwd packet length max,init_win_bytes_backward,init_win_bytes_forward,fwd packet length max,fwd packet length mean,fwd header length,flow duration,flow bytes/s,bwd packets/s,...,flow iat min,act_data_pkt_fwd,min_seg_size_forward,down/up ratio,fin flag count,idle std,urg flag count,psh flag count,active std,ack flag count
136684,76,326,-1,-1,38,38.000000,64,299,2.434783e+06,6688.963211,...,3,1,32,1,0,0.0,0,0,0.0,0
239336,12,0,29200,8192,6,4.000000,72,5788954,2.072913e+00,0.172743,...,93,2,20,0,0,0.0,0,1,0.0,0
610341,254,0,-1,30,217,84.666667,60,866,2.933025e+05,0.000000,...,4,2,20,0,0,0.0,0,0,0.0,1
1628679,887,2896,122,65535,405,32.851852,876,154725,3.467119e+05,219.744708,...,1,6,32,1,0,0.0,0,1,0.0,0
94758,1375,6,0,229,1375,1375.000000,20,9496466,1.479498e+02,0.631814,...,1,0,20,6,0,0.0,1,0,0.0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
141801,26,7215,229,8192,20,8.666667,72,15225,7.640722e+05,328.407225,...,3,2,20,1,0,0.0,0,1,0.0,0
1976239,1086,482,62,8192,1080,362.000000,72,241745,6.486173e+03,8.273180,...,119,2,20,0,0,0.0,0,1,0.0,0
1366464,37,0,-1,121,37,18.500000,64,3,1.230000e+07,0.000000,...,3,0,32,0,0,0.0,0,0,0.0,1
151255,26,8760,229,8192,20,8.666667,72,996314,1.167002e+04,4.014799,...,82,2,20,1,0,0.0,0,1,0.0,0


In [121]:
# identify highly skewed features on X_train (Skewness > 2.0)
skewness_scores = X_train_sub.skew().abs()
high_skew_cols = skewness_scores[skewness_scores > 2].index.tolist()
print(f" Applied Log1p transform to {len(high_skew_cols)} highly skewed features.")

 Applied Log1p transform to 32 highly skewed features.


In [122]:
all_skewness = X_train_sub.skew().abs()
high_skew_cols = all_skewness[all_skewness > 2].index.tolist()
escaped_cols = all_skewness[all_skewness <= 2].index.tolist()

print(f" (Skew > 2): {len(high_skew_cols)}")
print(f" (Skew <= 2): {len(escaped_cols)}\n")
print("(Escaped Columns):")

report_data = []
for col in escaped_cols:
    max_val = X_train_sub[col].max()
    min_val = X_train_sub[col].min()
    skew_val = all_skewness[col]

    report_data.append(
        {
            "Column Name": col,
            "Skewness": round(skew_val, 4),
            "Max Value": max_val,
            "Min Value": min_val,
        }
    )

escaped_df = pd.DataFrame(report_data).sort_values(by="Max Value", ascending=False)
print(escaped_df.to_string(index=False, float_format="{:,.2f}".format))

 (Skew > 2): 32
 (Skew <= 2): 3

(Escaped Columns):
   Column Name  Skewness  Max Value  Min Value
 flow duration      1.96  119999997        -12
psh flag count      0.89          1          0
ack flag count      0.81          1          0


In [ ]:
# forced_log_cols = [
#     col
#     for col in X_train_sub.columns
#     if any(
#         k in col.lower()
#         for k in [
#             "bytes",
#             "packets",
#             "duration",
#             "iat",
#             "length",
#             "size",
#             "active",
#             "idle",
#         ]
#     )
# ]

# final_log_cols = list(set(high_skew_cols + forced_log_cols))

In [ ]:
#  Apply Log1p (log(1+x)) transformation to stabilize variance
# for col in final_log_cols:
#     X_train_sub[col] = np.log1p(X_train_sub[col].clip(lower=0))
#     X_val_sub[col] = np.log1p(X_val_sub[col].clip(lower=0))
#     X_test_sub[col] = np.log1p(X_test_sub[col].clip(lower=0))

In [126]:
print("max num is: ", np.nanmax(X_train_sub.values))

max num is:  21.451297419842717


In [ ]:
# Fit RobustScaler strictly on X_train and transform all partitions
scaler = RobustScaler()
X_train_scaled = scaler.fit_transform(X_train_sub)
X_val_scaled = scaler.transform(X_val_sub)
X_test_scaled = scaler.transform(X_test_sub)

In [128]:
print(" Preprocessing and Robust Scaling completed successfully!")
print(f" - Final Scaled Train Shape: {X_train_scaled.shape}")
print(f" - Any remaining NaNs in Train: {np.isnan(X_train_scaled).any()}")
print(f" - Any remaining Infs in Train: {np.isinf(X_train_scaled).any()}")
print("max num is: ", np.nanmax(X_train_scaled))

 Preprocessing and Robust Scaling completed successfully!
 - Final Scaled Train Shape: (1764559, 35)
 - Any remaining NaNs in Train: False
 - Any remaining Infs in Train: False
max num is:  18.5159909328476


# scaling with QuantileTransformer

In [ ]:
from sklearn.preprocessing import QuantileTransformer

# MI-Boruta V2 
print(" Starting MI-Boruta V2...")

# 1. الاستعانة بالميزات التي اختارها MI-Boruta فقط
# تأكد أن المتغير selected_cols يحتوي على الـ 35 ميزة التي تم استخراجها سابقاً
X_train_mi = X_train_prep[selected_cols].copy()
X_val_mi = X_val_prep[selected_cols].copy()
X_test_mi = X_test_prep[selected_cols].copy()

# سد الفراغات بالوسيط كإجراء أمان لمنع أي انهيار بسبب Infs/NaNs
train_medians = X_train_mi.median().fillna(0)
X_train_mi.fillna(train_medians, inplace=True)
X_val_mi.fillna(train_medians, inplace=True)
X_test_mi.fillna(train_medians, inplace=True)

# 2. استخدام نفس السلاح: QuantileTransformer
print("\n Applying QuantileTransformer to MI features...")
qt_mi = QuantileTransformer(output_distribution="normal", random_state=42)
X_train_mi_qt = qt_mi.fit_transform(X_train_mi)
X_val_mi_qt = qt_mi.transform(X_val_mi)
X_test_mi_qt = qt_mi.transform(X_test_mi)

 Starting MI-Boruta V2 (Fair Rematch)...

📊 Applying QuantileTransformer to MI features...


# training

In [62]:
# from src.evaluation import convert_iforest_scores, iforest_predictions_to_binary, print_metrics_report, plot_roc_and_pr, log_experiment

# scores = convert_iforest_scores(iso_model, X_test)          # higher = more anomalous
# preds  = iforest_predictions_to_binary(iso_model, X_test)   # 1 = anomaly, 0 = normal

# metrics = print_metrics_report(y_test, preds, scores, model_name="Isolation Forest")
# plot_roc_and_pr(y_test, scores, model_name="Isolation Forest")

# log_experiment("../experiments/results_log.csv", "IsolationForest",
#                 params={"n_estimators": 200, "contamination": 0.03},
#                 metrics=metrics)

In [ ]:
from sklearn.ensemble import IsolationForest
from sklearn.metrics import (
    auc,
    classification_report,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
)
# STEP 4: Isolation Forest Training & Validation Threshold Optimization

# 1. Train strictly on BENIGN baseline traffic (y_train == 0)
print(" Subsetting BENIGN baseline traffic for training...")
benign_mask = np.array(y_train) == 0
X_train_benign = X_train_scaled[benign_mask]

print(
    f" - Total Train Samples: {X_train_scaled.shape[0]:,} | BENIGN Baseline Samples: {X_train_benign.shape[0]:,}"
)

# 2. Initialize Isolation Forest with tuned hyperparameters
iso_model = IsolationForest(
    n_estimators=400,  # 400 trees for stable score distribution
    max_samples=512,  # 512 samples per tree to optimize split resolution
    max_features=0.8,  # Subsample features for diversity
    contamination="auto",
    random_state=42,
    n_jobs=-1,
)

iso_model.fit(X_train_benign)

# 3. Calculate continuous anomaly scores on Validation Set (-decision_function)
val_scores = -iso_model.decision_function(X_val_scaled)

# 4. Compute Precision-Recall Curve & PR-AUC on Validation Set
precisions, recalls, thresholds = precision_recall_curve(y_val, val_scores)
val_pr_auc = auc(recalls, precisions)
val_roc_auc = roc_auc_score(y_val, val_scores)

print("\n Validation Global Curve Metrics:")
print(f" - Validation PR-AUC (AUC-PR): {val_pr_auc:.4f}")
print(f" - Validation ROC-AUC:         {val_roc_auc:.4f}")

# 5. Search for Optimal Threshold on PR-Curve (Maximizing F1-Score or Target SLA)
print("\n Searching for Optimal Decision Threshold on PR-Curve...")
best_thresh = None
best_f1 = 0.0
target_thresh = None
target_best_f1 = 0.0

for p, r, thresh in zip(precisions[:-1], recalls[:-1], thresholds):
    f1 = (2 * p * r) / (p + r) if (p + r) > 0 else 0.0

    # Track global highest F1 threshold
    if f1 > best_f1:
        best_f1 = f1
        best_thresh = thresh

    # Track specific SLA target threshold (Precision >= 85% & Recall >= 80%)
    if p >= 0.85 and r >= 0.80:
        if target_thresh is None or f1 > target_best_f1:
            target_best_f1 = f1
            target_thresh = thresh

# Use SLA threshold if met; otherwise fallback to maximum F1 threshold
selected_thresh = target_thresh if target_thresh is not None else best_thresh
val_preds = (val_scores >= selected_thresh).astype(int)

print(f"\n Optimal Frozen Threshold Selected: {selected_thresh:.4f}")
print(f" - Validation F1-Score:  {f1_score(y_val, val_preds, zero_division=0):.4f}")
print(
    f" - Validation Precision: {precision_score(y_val, val_preds, zero_division=0):.4f}"
)
print(f" - Validation Recall:    {recall_score(y_val, val_preds, zero_division=0):.4f}")

print("\n Validation Classification Report:")
print(
    classification_report(
        y_val, val_preds, target_names=["BENIGN (0)", "Attack (1)"], digits=4
    )
)

# STEP 5: Final Independent Evaluation on Unseen Test Set

print("\n Evaluating on Unseen Test Set using Frozen Validation Threshold...")
test_scores = -iso_model.decision_function(X_test_scaled)
test_preds = (test_scores >= selected_thresh).astype(int)

# Test Metrics
test_precisions, test_recalls, _ = precision_recall_curve(y_test, test_scores)
test_pr_auc = auc(test_recalls, test_precisions)
test_roc_auc = roc_auc_score(y_test, test_scores)

print("\n Final Test Set Benchmark Metrics:")
print(f" - Test PR-AUC (AUC-PR): {test_pr_auc:.4f}")
print(f" - Test ROC-AUC:         {test_roc_auc:.4f}")
print(f" - Test F1-Score:        {f1_score(y_test, test_preds, zero_division=0):.4f}")
print(
    f" - Test Precision:       {precision_score(y_test, test_preds, zero_division=0):.4f}"
)
print(
    f" - Test Recall:          {recall_score(y_test, test_preds, zero_division=0):.4f}"
)

print("\n    Test Set Classification Report:")
print(
    classification_report(
        y_test, test_preds, target_names=["BENIGN (0)", "Attack (1)"], digits=4
    )
)

 Subsetting BENIGN baseline traffic for training...
 - Total Train Samples: 1,764,559 | BENIGN Baseline Samples: 1,466,540

 Validation Global Curve Metrics:
 - Validation PR-AUC (AUC-PR): 0.4398
 - Validation ROC-AUC:         0.8776

 Searching for Optimal Decision Threshold on PR-Curve...

 Optimal Frozen Threshold Selected: 0.0197
 - Validation F1-Score:  0.6223
 - Validation Precision: 0.5492
 - Validation Recall:    0.7178

 Validation Classification Report:
              precision    recall  f1-score   support

  BENIGN (0)     0.9388    0.8803    0.9086    314258
  Attack (1)     0.5492    0.7178    0.6223     63861

    accuracy                         0.8528    378119
   macro avg     0.7440    0.7990    0.7655    378119
weighted avg     0.8730    0.8528    0.8603    378119


🧪 Evaluating on Unseen Test Set using Frozen Validation Threshold...

 Final Test Set Benchmark Metrics:
 - Test PR-AUC (AUC-PR): 0.4421
 - Test ROC-AUC:         0.8786
 - Test F1-Score:        0.6238
 - 

In [ ]:
# MI + quantile
from sklearn.ensemble import IsolationForest
from sklearn.metrics import (
    auc,
    classification_report,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
)

# 3. التدريب على الـ BENIGN فقط بمعاملات تناسب 35 ميزة
benign_mask = np.array(y_train) == 0
X_train_benign_mi = X_train_mi_qt[benign_mask]

# تم تعديل المعاملات الهندسية لتعطي الشجرة فرصة فحص 80% من الـ 35 ميزة
iso_mi_v2 = IsolationForest(
    n_estimators=100,  # رفعنا عدد الأشجار لاستيعاب الميزات الأكثر
    max_samples=0.9612,  # نفس العينة الضخمة التي استخدمتها ورقة PCA
    max_features=0.80,  # 80% من 35 ميزة = حوالي 28 ميزة لكل شجرة
    contamination="auto",
    random_state=42,
    n_jobs=-1,
)

print("\n Training Isolation Forest on 35 MI features...")
iso_mi_v2.fit(X_train_benign_mi)

# 4. البحث الحر عن أفضل Threshold (Maximizing F1-Score)
val_scores_mi = -iso_mi_v2.decision_function(X_val_mi_qt)
precisions, recalls, thresholds = precision_recall_curve(y_val, val_scores_mi)

best_f1_mi = 0.0
best_thresh_mi = None

for p, r, thresh in zip(precisions[:-1], recalls[:-1], thresholds):
    f1 = (2 * p * r) / (p + r) if (p + r) > 0 else 0.0
    if f1 > best_f1_mi:
        best_f1_mi = f1
        best_thresh_mi = thresh

val_pr_auc_mi = auc(recalls, precisions)
val_roc_auc_mi = roc_auc_score(y_val, val_scores_mi)

print(f"\n Validation Optimal Threshold Search:")
print(f" - Max Validation F1-Score:  {best_f1_mi:.4f}")
print(f" - Optimal Frozen Threshold: {best_thresh_mi:.4f}")
print(f" - Validation PR-AUC:        {val_pr_auc_mi:.4f}")
print(f" - Validation ROC-AUC:       {val_roc_auc_mi:.4f}")

# 5. التقييم النهائي على Test Set
test_scores_mi = -iso_mi_v2.decision_function(X_test_mi_qt)
test_preds_mi = (test_scores_mi >= best_thresh_mi).astype(int)

test_precisions, test_recalls, _ = precision_recall_curve(y_test, test_scores_mi)
test_pr_auc_mi = auc(test_recalls, test_precisions)
test_roc_auc_mi = roc_auc_score(y_test, test_scores_mi)

print("\n" + "=" * 65)
print(" FINAL TEST SET BENCHMARK RESULTS (MI-BORUTA V2)")
print("=" * 65)
print(f" - Test ROC-AUC:   {test_roc_auc_mi:.4f}")
print(f" - Test PR-AUC:    {test_pr_auc_mi:.4f}")
print(
    f" - Test Precision: {precision_score(y_test, test_preds_mi, zero_division=0):.4f}"
)
print(f" - Test Recall:    {recall_score(y_test, test_preds_mi, zero_division=0):.4f}")
print(f" - Test F1-Score:  {f1_score(y_test, test_preds_mi, zero_division=0):.4f}")
print("=" * 65)

print("\n Detailed Classification Report:")
print(
    classification_report(
        y_test, test_preds_mi, target_names=["BENIGN (0)", "Attack (1)"], digits=4
    )
)


 Training Isolation Forest on 35 MI features...

 Validation Optimal Threshold Search:
 - Max Validation F1-Score:  0.7835
 - Optimal Frozen Threshold: -0.0228
 - Validation PR-AUC:        0.8579
 - Validation ROC-AUC:       0.9657

 FINAL TEST SET BENCHMARK RESULTS (MI-BORUTA V2)
 - Test ROC-AUC:   0.9665
 - Test PR-AUC:    0.8627
 - Test Precision: 0.8238
 - Test Recall:    0.7488
 - Test F1-Score:  0.7845

 Detailed Classification Report:
              precision    recall  f1-score   support

  BENIGN (0)     0.9499    0.9674    0.9586    314259
  Attack (1)     0.8238    0.7488    0.7845     63861

    accuracy                         0.9305    378120
   macro avg     0.8868    0.8581    0.8715    378120
weighted avg     0.9286    0.9305    0.9292    378120



In [95]:
# (Experimentation Engine)
def run_anomaly_experiment(
    exp_name,
    X_tr,
    X_va,
    X_te,
    y_tr,
    y_va,
    y_te,
    scaler,
    use_pca=False,
    n_components=15,
    iso_params=None,
):
        
    print(f"\n Running: {exp_name}")
    if isinstance(scaler, RobustScaler):
        print("Applying Log1p transform automatically")
        X_tr, X_va, X_te = apply_log_transform(X_tr, X_va, X_te)

    # 1. Scaling
    X_tr_scaled = scaler.fit_transform(X_tr)
    X_va_scaled = scaler.transform(X_va)
    X_te_scaled = scaler.transform(X_te)

    # 2. Optional PCA
    if use_pca:
        pca = PCA(n_components=n_components, random_state=42)
        X_tr_final = pca.fit_transform(X_tr_scaled)
        X_va_final = pca.transform(X_va_scaled)
        X_te_final = pca.transform(X_te_scaled)
    else:
        X_tr_final, X_va_final, X_te_final = X_tr_scaled, X_va_scaled, X_te_scaled

    # 3. Semi-Supervised Training (BENIGN only)
    benign_mask = np.array(y_tr) == 0
    X_tr_benign = X_tr_final[benign_mask]

    model = IsolationForest(**iso_params)
    model.fit(X_tr_benign)

    # 4. F1-Max Threshold Search on Validation Set
    val_scores = -model.decision_function(X_va_final)
    precisions, recalls, thresholds = precision_recall_curve(y_va, val_scores)

    best_f1, best_thresh = 0.0, None
    for p, r, thresh in zip(precisions[:-1], recalls[:-1], thresholds):
        f1 = (2 * p * r) / (p + r) if (p + r) > 0 else 0.0
        if f1 > best_f1:
            best_f1 = f1
            best_thresh = thresh

    # 5. Final Evaluation on Test Set
    test_scores = -model.decision_function(X_te_final)
    test_preds = (test_scores >= best_thresh).astype(int)

    metrics = {
        "Experiment": exp_name,
        "Test F1-Score": f1_score(y_te, test_preds, zero_division=0),
        "Test Precision": precision_score(y_te, test_preds, zero_division=0),
        "Test Recall": recall_score(y_te, test_preds, zero_division=0),
        "Test ROC-AUC": roc_auc_score(y_te, test_scores),
        "Optimal Thresh": best_thresh,
    }
    return metrics

In [96]:
results_list = []

# أ. بيانات MI-Boruta (الميزات الـ 35 التي حددناها مسبقاً)
X_tr_mi = (
    X_train_prep[selected_cols].copy().fillna(X_train_prep[selected_cols].median())
)
X_va_mi = X_val_prep[selected_cols].copy().fillna(X_train_prep[selected_cols].median())
X_te_mi = X_test_prep[selected_cols].copy().fillna(X_train_prep[selected_cols].median())

# ب. البيانات الخام بالكامل (لكل الميزات بدون فلترة)
X_tr_all = X_train_prep.copy().fillna(X_train_prep.median())
X_va_all = X_val_prep.copy().fillna(X_train_prep.median())
X_te_all = X_test_prep.copy().fillna(X_train_prep.median())

In [ ]:
# 1. تجربة Verkerken (PCA على كل البيانات الخام + Quantile)
res1 = run_anomaly_experiment(
    exp_name="1. Pure PCA (All Features + Quantile)",
    X_tr=X_tr_all,
    X_va=X_va_all,
    X_te=X_te_all,
    y_tr=y_train,
    y_va=y_val,
    y_te=y_test,
    scaler=QuantileTransformer(output_distribution="normal", random_state=42),
    use_pca=True,
    n_components=15,
    iso_params={
        "n_estimators": 58,
        "max_samples": 0.9612,
        "max_features": 0.3672,
        "random_state": 42,
        "n_jobs": -1,
    },
)
results_list.append(res1)

# 2. تجربة MI-Boruta مع Quantile (مقارنة عادلة مع Verkerken)
res2 = run_anomaly_experiment(
    exp_name="2. MI-Boruta + Quantile (No PCA)",
    X_tr=X_tr_mi,
    X_va=X_va_mi,
    X_te=X_te_mi,
    y_tr=y_train,
    y_va=y_val,
    y_te=y_test,
    scaler=QuantileTransformer(output_distribution="normal", random_state=42),
    use_pca=False,
    iso_params={
        "n_estimators": 100,
        "max_samples": 0.9612,
        "max_features": 0.80,
        "random_state": 42,
        "n_jobs": -1,
    },
)
results_list.append(res2)

# 3. تجربة MI-Boruta مع RobustScaler (البايب لاين الأمني الأصلي الخاص بك)
res3 = run_anomaly_experiment(
    exp_name="3. MI-Boruta + RobustScaler (Production)",
    X_tr=X_tr_mi,
    X_va=X_va_mi,
    X_te=X_te_mi,
    y_tr=y_train,
    y_va=y_val,
    y_te=y_test,
    scaler=RobustScaler(),
    use_pca=False,
    iso_params={
        "n_estimators": 100,
        "max_samples": 0.9612,
        "max_features": 0.80,
        "random_state": 42,
        "n_jobs": -1,
    },
)
results_list.append(res3)


 Running: 1. Pure PCA (All Features + Quantile)

 Running: 2. MI-Boruta + Quantile (No PCA)

 Running: 3. MI-Boruta + RobustScaler (Production)
Applying Log1p transform automatically
 Applied Log1p transform to 26 highly skewed features.


In [98]:
# طباعة الجدول النهائي
print("\n" + "=" * 75)
print(" FINAL EXPERIMENTATION LEADERBOARD")
print("=" * 75)
leaderboard_df = pd.DataFrame(results_list)
leaderboard_df = leaderboard_df.sort_values(
    by="Test F1-Score", ascending=False
).reset_index(drop=True)
print(leaderboard_df.to_string(index=False, float_format="{:.4f}".format))
print("=" * 75)


 FINAL EXPERIMENTATION LEADERBOARD
                              Experiment  Test F1-Score  Test Precision  Test Recall  Test ROC-AUC  Optimal Thresh
   1. Pure PCA (All Features + Quantile)         0.7966          0.7032       0.9185        0.9697         -0.0555
        2. MI-Boruta + Quantile (No PCA)         0.7477          0.7368       0.7590        0.9503         -0.0425
3. MI-Boruta + RobustScaler (Production)         0.5812          0.4401       0.8557        0.8583         -0.0094


In [115]:
# (The Mega Engine)
def run_anomaly_experiment(
    exp_name,
    X_tr,
    X_va,
    X_te,
    y_tr,
    y_va,
    y_te,
    scaler,
    iso_params,
    use_pca=False,
    n_components=15,
    fs_metadata=None,
):
    # أ. معالجة اللوغاريتم تلقائياً لو كان المقيّس هو RobustScaler
    if isinstance(scaler, RobustScaler):
        X_tr, X_va, X_te = apply_log_transform(X_tr, X_va, X_te)

    # ب. خطوة التحجيم (Scaling)
    X_tr_scaled = scaler.fit_transform(X_tr)
    X_va_scaled = scaler.transform(X_va)
    X_te_scaled = scaler.transform(X_te)

    # ج. تطبيق الـ PCA الاختياري (في حال أراد النظام تجربة الـ Baseline)
    if use_pca:
        pca = PCA(n_components=n_components, random_state=42)
        X_tr_final = pca.fit_transform(X_tr_scaled)
        X_va_final = pca.transform(X_va_scaled)
        X_te_final = pca.transform(X_te_scaled)
    else:
        X_tr_final, X_va_final, X_te_final = X_tr_scaled, X_va_scaled, X_te_scaled

    # د. التدريب شبه الخاضع للإشراف (على الترافيك السليم BENIGN فقط)
    benign_mask = np.array(y_tr) == 0
    model = IsolationForest(**iso_params)
    model.fit(X_tr_final[benign_mask])

    # هـ. البحث الديناميكي عن أفضل Threshold لتحقيق أقصى F1-Score
    val_scores = -model.decision_function(X_va_final)
    precisions, recalls, thresholds = precision_recall_curve(y_va, val_scores)

    best_f1, best_thresh = 0.0, None
    for p, r, thresh in zip(precisions[:-1], recalls[:-1], thresholds):
        f1 = (2 * p * r) / (p + r) if (p + r) > 0 else 0.0
        if f1 > best_f1:
            best_f1, best_thresh = f1, thresh

    # و. التقييم النهائي الحقيقي على مجموعة الاختبار (Test Set)
    test_scores = -model.decision_function(X_te_final)
    test_preds = (test_scores >= best_thresh).astype(int)
    test_precisions, test_recalls, _ = precision_recall_curve(y_te, test_scores)

    # ز. تجميع كل المقاييس والمعاملات في قاموس موحد
    metrics = {
        "Experiment": exp_name,
        "Test F1": f1_score(y_te, test_preds, zero_division=0),
        "Test Precision": precision_score(y_te, test_preds, zero_division=0),
        "Test Recall": recall_score(y_te, test_preds, zero_division=0),
        "PR-AUC": auc(test_recalls, test_precisions),
        "ROC-AUC": roc_auc_score(y_te, test_scores),
        "Optimal_Thresh": best_thresh,
    }

    # دمج معلومات الـ Hyperparameters في الجدول
    for key, value in iso_params.items():
        metrics[f"Param_{key}"] = value

    # دمج معلومات الـ Feature Selection / PCA
    if fs_metadata:
        for key, value in fs_metadata.items():
            metrics[f"Setup_{key}"] = value

    return metrics

In [116]:
results_list = []

# أ. تجهيز سيناريو MI-Boruta (الميزات المختارة)
X_tr_mi = (
    X_train_prep[selected_cols].copy().fillna(X_train_prep[selected_cols].median())
)
X_va_mi = X_val_prep[selected_cols].copy().fillna(X_train_prep[selected_cols].median())
X_te_mi = X_test_prep[selected_cols].copy().fillna(X_train_prep[selected_cols].median())

# ب. تجهيز سيناريو PCA (كل البيانات الخام بدون فلترة ميزات)
X_tr_all = X_train_prep.copy().fillna(X_train_prep.median())
X_va_all = X_val_prep.copy().fillna(X_train_prep.median())
X_te_all = X_test_prep.copy().fillna(X_train_prep.median())

In [117]:
# ج. شبكة الاحتمالات لـ Hyperparameters الخاصة بـ Isolation Forest
param_grid = {
    "n_estimators": [100, 150],
    "max_samples": [256, 512, 0.9612],
    "max_features": [0.6, 0.8, 1.0],
    "random_state": [42],
    "n_jobs": [-1],
}
grid = list(ParameterGrid(param_grid))

# د. قائمة المقيّسات المراد اختبارها
scalers_to_test = [
    RobustScaler(),
    QuantileTransformer(output_distribution="normal", random_state=42),
]

total_experiments = len(grid) * len(scalers_to_test) + len(
    grid
)  
# (MI/Boruta variants + PCA baseline)
print(
    f"\n Starting Mega GridSearch Pipeline! Total Planned Experiments: ~{total_experiments}\n")


 Starting Mega GridSearch Pipeline! Total Planned Experiments: ~54



In [118]:
# المرحلة الأولى: اختبار سيناريوهات MI-Boruta مع مختلف Scalers & Hyperparameters
for scaler in scalers_to_test:
    scaler_name = scaler.__class__.__name__

    for i, params in enumerate(grid):
        exp_name = f"MI+Boruta | {scaler_name} | Run_{i+1}"
        fs_info = {
            "Method": "MI+Boruta",
            "Features_Count": len(selected_cols),
            "Scaler": scaler_name,
        }
        X_tr_clean = X_tr_mi.copy()
        X_va_clean = X_va_mi.copy()
        X_te_clean = X_te_mi.copy()

        res = run_anomaly_experiment(
            exp_name=exp_name,
            X_tr=X_tr_mi,
            X_va=X_va_mi,
            X_te=X_te_mi,
            y_tr=y_train,
            y_va=y_val,
            y_te=y_test,
            scaler=scaler,
            iso_params=params,
            use_pca=False,
            fs_metadata=fs_info,
        )
        results_list.append(res)
        print(f" Finished: {exp_name} --> F1-Score: {res['Test F1']:.4f}")

 [Log Transform] Applying log1p to 27 skewed features.
 Finished: MI+Boruta | RobustScaler | Run_1 --> F1-Score: 0.5483
 [Log Transform] Applying log1p to 27 skewed features.
 Finished: MI+Boruta | RobustScaler | Run_2 --> F1-Score: 0.5411
 [Log Transform] Applying log1p to 27 skewed features.
 Finished: MI+Boruta | RobustScaler | Run_3 --> F1-Score: 0.5829
 [Log Transform] Applying log1p to 27 skewed features.
 Finished: MI+Boruta | RobustScaler | Run_4 --> F1-Score: 0.5624
 [Log Transform] Applying log1p to 27 skewed features.
 Finished: MI+Boruta | RobustScaler | Run_5 --> F1-Score: 0.8334
 [Log Transform] Applying log1p to 27 skewed features.
 Finished: MI+Boruta | RobustScaler | Run_6 --> F1-Score: 0.8365
 [Log Transform] Applying log1p to 27 skewed features.
 Finished: MI+Boruta | RobustScaler | Run_7 --> F1-Score: 0.5405
 [Log Transform] Applying log1p to 27 skewed features.
 Finished: MI+Boruta | RobustScaler | Run_8 --> F1-Score: 0.5700
 [Log Transform] Applying log1p to 27 sk

In [119]:
# Baseline (Pure PCA + Quantile)
for i, params in enumerate(grid):
    exp_name = f"Pure_PCA | Quantile | Run_{i+1}"
    fs_info = {
        "Method": "Pure_PCA",
        "Features_Count": 15,
        "Scaler": "QuantileTransformer",
    }

    res = run_anomaly_experiment(
        exp_name=exp_name,
        X_tr=X_tr_all,
        X_va=X_va_all,
        X_te=X_te_all,
        y_tr=y_train,
        y_va=y_val,
        y_te=y_test,
        scaler=QuantileTransformer(output_distribution="normal", random_state=42),
        iso_params=params,
        use_pca=True,
        n_components=15,
        fs_metadata=fs_info,
    )
    results_list.append(res)
    print(f" Finished: {exp_name} --> F1-Score: {res['Test F1']:.4f}")

 Finished: Pure_PCA | Quantile | Run_1 --> F1-Score: 0.5295
 Finished: Pure_PCA | Quantile | Run_2 --> F1-Score: 0.5358
 Finished: Pure_PCA | Quantile | Run_3 --> F1-Score: 0.5564
 Finished: Pure_PCA | Quantile | Run_4 --> F1-Score: 0.5623
 Finished: Pure_PCA | Quantile | Run_5 --> F1-Score: 0.8177
 Finished: Pure_PCA | Quantile | Run_6 --> F1-Score: 0.8160
 Finished: Pure_PCA | Quantile | Run_7 --> F1-Score: 0.5041
 Finished: Pure_PCA | Quantile | Run_8 --> F1-Score: 0.5284
 Finished: Pure_PCA | Quantile | Run_9 --> F1-Score: 0.5735
 Finished: Pure_PCA | Quantile | Run_10 --> F1-Score: 0.5777


KeyboardInterrupt: 

In [120]:
print("\n" + "=" * 100)
print(" ULTIMATE EXPERIMENTATION LEADERBOARD (Sorted by F1-Score)")
print("=" * 100)

leaderboard_df = pd.DataFrame(results_list)
leaderboard_df = leaderboard_df.sort_values(by="Test F1", ascending=False).reset_index(
    drop=True
)

# طباعة أعظم نتيجة (الفائز الأول)
best_run = leaderboard_df.iloc[0]
print("\n" + "🌟" * 40)
print("🥇 THE ABSOLUTE WINNING CONFIGURATION:")
print("🌟" * 40)
for col, val in best_run.items():
    print(f" - {col}: {val}")
print("🌟" * 40 + "\n")

# عرض أفضل 10 تجارب في جدول نظيف
display(leaderboard_df.head(10))


 ULTIMATE EXPERIMENTATION LEADERBOARD (Sorted by F1-Score)

🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟
🥇 THE ABSOLUTE WINNING CONFIGURATION:
🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟
 - Experiment: MI+Boruta | RobustScaler | Run_6
 - Test F1: 0.8365477266999718
 - Test Precision: 0.7942099559483765
 - Test Recall: 0.8836535600757897
 - PR-AUC: 0.8744035006265719
 - ROC-AUC: 0.9769159046819877
 - Optimal_Thresh: -0.0329427196307403
 - Param_max_features: 0.6
 - Param_max_samples: 0.9612
 - Param_n_estimators: 150
 - Param_n_jobs: -1
 - Param_random_state: 42
 - Setup_Method: MI+Boruta
 - Setup_Features_Count: 27
 - Setup_Scaler: RobustScaler
🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟



,Experiment,Test F1,Test Precision,Test Recall,PR-AUC,ROC-AUC,Optimal_Thresh,Param_max_features,Param_max_samples,Param_n_estimators,Param_n_jobs,Param_random_state,Setup_Method,Setup_Features_Count,Setup_Scaler
0,MI+Boruta | RobustScaler | Run_6,0.836548,0.794210,0.883654,0.874404,0.976916,-0.032943,0.6,0.9612,150,-1,42,MI+Boruta,27,RobustScaler
1,MI+Boruta | RobustScaler | Run_17,0.834612,0.857126,0.813251,0.875560,0.976443,-0.015479,1.0,0.9612,100,-1,42,MI+Boruta,27,RobustScaler
2,MI+Boruta | RobustScaler | Run_18,0.834279,0.875239,0.796981,0.876906,0.976160,-0.010080,1.0,0.9612,150,-1,42,MI+Boruta,27,RobustScaler
3,MI+Boruta | RobustScaler | Run_5,0.833373,0.855123,0.812703,0.870895,0.975723,-0.015621,0.6,0.9612,100,-1,42,MI+Boruta,27,RobustScaler
4,MI+Boruta | RobustScaler | Run_12,0.826993,0.809459,0.845305,0.876634,0.975801,-0.026441,0.8,0.9612,150,-1,42,MI+Boruta,27,RobustScaler
5,Pure_PCA | Quantile | Run_5,0.817731,0.785375,0.852868,0.875800,0.970056,-0.043168,0.6,0.9612,100,-1,42,Pure_PCA,15,QuantileTransformer
6,Pure_PCA | Quantile | Run_6,0.816003,0.764475,0.874978,0.868848,0.969061,-0.048491,0.6,0.9612,150,-1,42,Pure_PCA,15,QuantileTransformer
7,MI+Boruta | RobustScaler | Run_11,0.813472,0.789554,0.838884,0.870261,0.973816,-0.029943,0.8,0.9612,100,-1,42,MI+Boruta,27,RobustScaler
8,MI+Boruta | QuantileTransformer | Run_5,0.793567,0.726263,0.874618,0.806900,0.958634,-0.051456,0.6,0.9612,100,-1,42,MI+Boruta,27,QuantileTransformer
9,MI+Boruta | QuantileTransformer | Run_6,0.788317,0.720775,0.869827,0.808921,0.959225,-0.050181,0.6,0.9612,150,-1,42,MI+Boruta,27,QuantileTransformer


In [121]:
print(" Initializing Modular GridSearch Pipeline...\n")


# RobustScaler
def apply_log_transform(x_tr, x_va, x_te):
    x_tr_log = x_tr.copy()
    x_va_log = x_va.copy()
    x_te_log = x_te.copy()

    skewness_scores = x_tr_log.skew().abs()
    high_skew_cols = skewness_scores[skewness_scores > 2].index.tolist()
    forced_keywords = [
        "bytes",
        "packets",
        "duration",
        "iat",
        "length",
        "size",
        "active",
        "idle",
    ]
    forced_log_cols = [
        col
        for col in x_tr_log.columns
        if any(k in col.lower() for k in forced_keywords)
    ]
    final_log_cols = list(set(high_skew_cols + forced_log_cols))

    for col in final_log_cols:
        x_tr_log[col] = np.log1p(x_tr_log[col].clip(lower=0))
        x_va_log[col] = np.log1p(x_va_log[col].clip(lower=0))
        x_te_log[col] = np.log1p(x_te_log[col].clip(lower=0))

    return x_tr_log, x_va_log, x_te_log


# ============================================================================== #
# 2. تحضير وتخزين المتغيرات (Feature Selection & Scaling Variables)
# ============================================================================== #
print("📦 Preparing and caching feature-selected & scaled variables...")

# أ. استخراج وتخزين بيانات الـ Feature Selection (MI + Boruta) في متغير
X_tr_mi = (
    X_train_prep[selected_cols].copy().fillna(X_train_prep[selected_cols].median())
)
X_va_mi = X_val_prep[selected_cols].copy().fillna(X_train_prep[selected_cols].median())
X_te_mi = X_test_prep[selected_cols].copy().fillna(X_test_prep[selected_cols].median())

# ب. تجهيز متغيرات التحجيم (Scaling Approaches) للـ MI + Boruta
# Approach 1: RobustScaler (مع Log مسبق)
X_tr_mi_log, X_va_mi_log, X_te_mi_log = apply_log_transform(X_tr_mi, X_va_mi, X_te_mi)
robust_scaler = RobustScaler()
X_tr_mi_robust = robust_scaler.fit_transform(X_tr_mi_log)
X_va_mi_robust = robust_scaler.transform(X_va_mi_log)
X_te_mi_robust = robust_scaler.transform(X_te_mi_log)

 Initializing Modular GridSearch Pipeline...

📦 Preparing and caching feature-selected & scaled variables...


In [ ]:
quantile_scaler = QuantileTransformer(output_distribution="normal", random_state=42)
X_tr_mi_quantile = quantile_scaler.fit_transform(X_tr_mi)
X_va_mi_quantile = quantile_scaler.transform(X_va_mi)
X_te_mi_quantile = quantile_scaler.transform(X_te_mi)

In [ ]:
X_tr_all = X_train_prep.copy().fillna(X_train_prep.median())
X_va_all = X_val_prep.copy().fillna(X_train_prep.median())
X_te_all = X_test_prep.copy().fillna(X_test_prep.median())

pca_scaler = QuantileTransformer(output_distribution="normal", random_state=42)
X_tr_all_scaled = pca_scaler.fit_transform(X_tr_all)
X_va_all_scaled = pca_scaler.transform(X_va_all)
X_te_all_scaled = pca_scaler.transform(X_te_all)

pca = PCA(n_components=15, random_state=42)
X_tr_pca = pca.fit_transform(X_tr_all_scaled)
X_va_pca = pca.transform(X_va_all_scaled)
X_te_pca = pca.transform(X_te_all_scaled)

print("\n✅ All variables are successfully initialized and stored!\n")


✅ All variables are successfully initialized and stored!



In [ ]:
# GridSearch
results_list = []

param_grid = {
    "n_estimators": [100, 150],
    "max_samples": [256, 512, 0.9612],
    "max_features": [0.6, 0.8, 1.0],
    "random_state": [42],
    "n_jobs": [-1],
}
grid = list(ParameterGrid(param_grid))

experiments_dict = {
    "MI+Boruta | RobustScaler": (X_tr_mi_robust, X_va_mi_robust, X_te_mi_robust),
    "MI+Boruta | QuantileTransformer": (
        X_tr_mi_quantile,
        X_va_mi_quantile,
        X_te_mi_quantile,
    ),
    "Pure_PCA (15 Comp) | Quantile": (X_tr_pca, X_va_pca, X_te_pca),
}

total_runs = len(grid) * len(experiments_dict)
print(
    f"🚀 Running GridSearch across all variables! Total Runs: {total_runs}\n" + "=" * 60
)

🚀 Running GridSearch across all variables! Total Runs: 54


In [125]:
for exp_title, (X_tr_var, X_va_var, X_te_var) in experiments_dict.items():
    print(f"\n Testing Strategy Variable: {exp_title}")

    for i, params in enumerate(grid):
        exp_name = f"{exp_title} | Run_{i+1}"

        # أ. التدريب شبه الخاضع للإشراف على الترافيك السليم (Benign) فقط
        benign_mask = np.array(y_train) == 0
        model = IsolationForest(**params)
        model.fit(X_tr_var[benign_mask])

        # ب. البحث عن أفضل Threshold بناءً على أعلى F1-Score
        val_scores = -model.decision_function(X_va_var)
        precisions, recalls, thresholds = precision_recall_curve(y_val, val_scores)

        best_f1, best_thresh = 0.0, None
        for p, r, thresh in zip(precisions[:-1], recalls[:-1], thresholds):
            f1 = (2 * p * r) / (p + r) if (p + r) > 0 else 0.0
            if f1 > best_f1:
                best_f1, best_thresh = f1, thresh

        # ج. التقييم النهائي الحقيقي على الـ Test Set وحساب المنحنيات
        test_scores = -model.decision_function(X_te_var)
        test_preds = (test_scores >= best_thresh).astype(int)
        test_precisions, test_recalls, _ = precision_recall_curve(y_test, test_scores)

        # د. تسجيل النتائج
        metrics = {
            "Experiment": exp_name,
            "Approach": exp_title,
            "Test F1": f1_score(y_test, test_preds, zero_division=0),
            "Test Precision": precision_score(y_test, test_preds, zero_division=0),
            "Test Recall": recall_score(y_test, test_preds, zero_division=0),
            "PR-AUC": auc(test_recalls, test_precisions),
            "ROC-AUC": roc_auc_score(y_test, test_scores),
            "Optimal_Thresh": best_thresh,
        }

        for key, value in params.items():
            metrics[f"Param_{key}"] = value

        results_list.append(metrics)
        print(f"   ✅ {exp_name} --> F1-Score: {metrics['Test F1']:.4f}")


 Testing Strategy Variable: MI+Boruta | RobustScaler
   ✅ MI+Boruta | RobustScaler | Run_1 --> F1-Score: 0.5483
   ✅ MI+Boruta | RobustScaler | Run_2 --> F1-Score: 0.5411
   ✅ MI+Boruta | RobustScaler | Run_3 --> F1-Score: 0.5829
   ✅ MI+Boruta | RobustScaler | Run_4 --> F1-Score: 0.5624
   ✅ MI+Boruta | RobustScaler | Run_5 --> F1-Score: 0.8334
   ✅ MI+Boruta | RobustScaler | Run_6 --> F1-Score: 0.8365
   ✅ MI+Boruta | RobustScaler | Run_7 --> F1-Score: 0.5405
   ✅ MI+Boruta | RobustScaler | Run_8 --> F1-Score: 0.5700
   ✅ MI+Boruta | RobustScaler | Run_9 --> F1-Score: 0.5812
   ✅ MI+Boruta | RobustScaler | Run_10 --> F1-Score: 0.5895
   ✅ MI+Boruta | RobustScaler | Run_11 --> F1-Score: 0.8135
   ✅ MI+Boruta | RobustScaler | Run_12 --> F1-Score: 0.8270
   ✅ MI+Boruta | RobustScaler | Run_13 --> F1-Score: 0.5405
   ✅ MI+Boruta | RobustScaler | Run_14 --> F1-Score: 0.5367
   ✅ MI+Boruta | RobustScaler | Run_15 --> F1-Score: 0.6075
   ✅ MI+Boruta | RobustScaler | Run_16 --> F1-Score: 0.

In [126]:
print("\n" + "=" * 110)
print(" MODULAR GRIDSEARCH LEADERBOARD (Sorted by F1-Score)")
print("=" * 110)

leaderboard_df = pd.DataFrame(results_list)
leaderboard_df = leaderboard_df.sort_values(by="Test F1", ascending=False).reset_index(
    drop=True
)

# طباعة الفائز المطلق
best_run = leaderboard_df.iloc[0]
print("\n" + "🌟" * 44)
print("🥇 THE ABSOLUTE WINNING CONFIGURATION:")
print("🌟" * 44)
for col, val in best_run.items():
    print(f" - {col}: {val}")
print("🌟" * 44 + "\n")

display(leaderboard_df.head(10))


 MODULAR GRIDSEARCH LEADERBOARD (Sorted by F1-Score)

🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟
🥇 THE ABSOLUTE WINNING CONFIGURATION:
🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟
 - Experiment: MI+Boruta | RobustScaler | Run_6
 - Approach: MI+Boruta | RobustScaler
 - Test F1: 0.8365477266999718
 - Test Precision: 0.7942099559483765
 - Test Recall: 0.8836535600757897
 - PR-AUC: 0.8744035006265719
 - ROC-AUC: 0.9769159046819877
 - Optimal_Thresh: -0.0329427196307403
 - Param_max_features: 0.6
 - Param_max_samples: 0.9612
 - Param_n_estimators: 150
 - Param_n_jobs: -1
 - Param_random_state: 42
🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟



,Experiment,Approach,Test F1,Test Precision,Test Recall,PR-AUC,ROC-AUC,Optimal_Thresh,Param_max_features,Param_max_samples,Param_n_estimators,Param_n_jobs,Param_random_state
0,MI+Boruta | RobustScaler | Run_6,MI+Boruta | RobustScaler,0.836548,0.794210,0.883654,0.874404,0.976916,-0.032943,0.6,0.9612,150,-1,42
1,MI+Boruta | RobustScaler | Run_17,MI+Boruta | RobustScaler,0.834612,0.857126,0.813251,0.875560,0.976443,-0.015479,1.0,0.9612,100,-1,42
2,MI+Boruta | RobustScaler | Run_18,MI+Boruta | RobustScaler,0.834279,0.875239,0.796981,0.876906,0.976160,-0.010080,1.0,0.9612,150,-1,42
3,MI+Boruta | RobustScaler | Run_5,MI+Boruta | RobustScaler,0.833373,0.855123,0.812703,0.870895,0.975723,-0.015621,0.6,0.9612,100,-1,42
4,MI+Boruta | RobustScaler | Run_12,MI+Boruta | RobustScaler,0.826993,0.809459,0.845305,0.876634,0.975801,-0.026441,0.8,0.9612,150,-1,42
5,Pure_PCA (15 Comp) | Quantile | Run_5,Pure_PCA (15 Comp) | Quantile,0.817731,0.785375,0.852868,0.875800,0.970056,-0.043168,0.6,0.9612,100,-1,42
6,Pure_PCA (15 Comp) | Quantile | Run_6,Pure_PCA (15 Comp) | Quantile,0.816003,0.764475,0.874978,0.868848,0.969061,-0.048491,0.6,0.9612,150,-1,42
7,MI+Boruta | RobustScaler | Run_11,MI+Boruta | RobustScaler,0.813472,0.789554,0.838884,0.870261,0.973816,-0.029943,0.8,0.9612,100,-1,42
8,Pure_PCA (15 Comp) | Quantile | Run_12,Pure_PCA (15 Comp) | Quantile,0.809591,0.750934,0.878189,0.856637,0.966753,-0.054961,0.8,0.9612,150,-1,42
9,Pure_PCA (15 Comp) | Quantile | Run_11,Pure_PCA (15 Comp) | Quantile,0.804691,0.755564,0.860650,0.859326,0.966723,-0.053476,0.8,0.9612,100,-1,42
